[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rpasquini/econometrics-book-notebooks/blob/main/es/ch4_inferencia_causal/6_synthetic_control_notebook.ipynb)

# Cuaderno de código: El Método de Control Sintético

Este cuaderno acompaña la sección [El Método de Control Sintético](6_synthetic_control.md). Los dashboards interactivos de esa sección sirven para *sentir* cómo se construyen los pesos y cómo funciona la inferencia por permutación moviendo controles; este cuaderno muestra cómo se resuelve el mismo problema en un **pipeline real de análisis**, combinando la mecánica del Dashboard 1 (construcción de pesos) y del Dashboard 3 (inferencia por permutación) en un flujo aplicado sobre dos conjuntos de datos:

1. **La misma simulación de Uber/Austin** que genera el Dashboard 1 — congelada en un `.csv` con semilla fija, para que el análisis sea reproducible.
2. **El caso de estudio clásico**: la Proposición 99 de California (Abadie, Diamond & Hainmueller, 2010) — el paper que introdujo el método de control sintético.

Para cada conjunto de datos implementamos primero una versión **desde cero** (para ver la mecánica exacta que hay detrás del estimador) y luego la misma idea usando `scpi_pkg`, una librería especializada en control sintético y sus intervalos de predicción.

## Citas

- Abadie, A., Diamond, A., & Hainmueller, J. (2010). *Synthetic Control Methods for Comparative Case Studies: Estimating the Effect of California's Tobacco Control Program.* Journal of the American Statistical Association, 105(490), 493–505.
- Los datos de Proposition 99 provienen del paquete de R `synthdid` (Arkhangelsky, Athey, Hirshberg, Imbens & Wager), licenciado BSD-3-Clause — permite redistribución y reuso con atribución. Repositorio: [synth-inference/synthdid](https://github.com/synth-inference/synthdid).
- Cattaneo, M. D., Feng, Y., Palomba, F., & Titiunik, R. (2025). *scpi: Uncertainty Quantification for Synthetic Control Estimators.* Journal of Statistical Software, 113(1), 1–38.
- Cattaneo, M. D., Feng, Y., & Titiunik, R. (2021). *Prediction Intervals for Synthetic Control Methods.* Journal of the American Statistical Association, 116(536), 1865–1880.
- (Para referencia futura) Abadie, A., Diamond, A., & Hainmueller, J. (2015). *Comparative Politics and the Synthetic Control Method.* American Journal of Political Science, 59(2), 495–510 — estudio de la reunificación alemana, cuyos datos ya están descargados en `data/` pero todavía no se usan en este cuaderno.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize

# scpi_pkg is not preinstalled on Colab -- installs quickly, safe to skip if already present
try:
    import scpi_pkg  # noqa: F401
except ImportError:
    %pip install -q scpi_pkg

from scpi_pkg.scdata import scdata
from scpi_pkg.scest import scest
from scpi_pkg.scpi import scpi

## Parte 1 — La simulación de Uber/Austin

> **Nota:** los datos de Uber/Austin usados a continuación son datos ficticios, generados por computadora con fines didácticos — no son datos reales de Uber. Permiten ilustrar el estimador con un proceso generador de datos conocido y controlable, antes de pasar al caso de estudio real de la Proposición 99.

`rides_thousands` son los viajes completados semanales (miles), en Austin y en un *pool* de 18 ciudades donantes. Las primeras 20 semanas son pre-intervención; el programa de incentivos a conductores empieza en la semana 21. Es el mismo dato que genera el Dashboard 1, congelado con la misma semilla aleatoria (`comparison_mode="optimized"`, `true_effect=10%`).

In [ ]:
uber = pd.read_csv('https://raw.githubusercontent.com/rpasquini/econometrics-book-notebooks/main/es/ch4_inferencia_causal/data/synthetic_control_uber_austin.csv')
uber.head()

### Estimador 1a — Pesos del control sintético, desde cero

Reproducimos exactamente la mecánica del Dashboard 1: 8 predictores por ciudad (4 variables demográficas + 4 observaciones rezagadas de viajes en las semanas 5, 10, 15 y 20), y resolvemos

$$\min_W (X_1 - X_0 W)'V(X_1-X_0W) \quad \text{s.a. } W\ge 0,\ \mathbf{1}'W=1$$

con $V$ la "selección simple" (ponderación por varianza inversa de cada predictor).

In [ ]:
T0 = 20
HISTORY_WEEKS = [5, 10, 15, 20]
DEMO_COLS = ["population_m", "income_k", "drivers_k", "lyft_share_pct"]

def build_predictor_matrix(df, cities):
    rows = []
    for city in cities:
        sub = df[df.city == city]
        demo = sub[DEMO_COLS].iloc[0].values
        hist = sub[sub.week.isin(HISTORY_WEEKS)].sort_values("week")["rides_thousands"].values
        rows.append(np.concatenate([demo, hist]))
    return np.array(rows).T  # (8, n_cities)

def solve_weights(x1, x0):
    """Constrained least squares: minimize ||x1 - x0 @ w||_V s.t. w >= 0, sum(w) = 1."""
    J = x0.shape[1]
    V = 1.0 / np.maximum(x0.var(axis=1), 1e-6)

    def loss(w):
        diff = x1 - x0 @ w
        return float(diff @ (V * diff))

    res = minimize(loss, x0=np.full(J, 1.0 / J), method="SLSQP",
                    bounds=[(0.0, 1.0)] * J,
                    constraints=[{"type": "eq", "fun": lambda w: w.sum() - 1.0}],
                    options={"maxiter": 300, "ftol": 1e-10})
    w = np.clip(res.x, 0, None)
    total = w.sum()
    return w / total if total > 0 else np.full(J, 1.0 / J)

donor_cities = sorted(uber.loc[uber.is_treated_unit == 0, "city"].unique())
X = build_predictor_matrix(uber, ["Austin"] + donor_cities)
w_uber = solve_weights(X[:, 0], X[:, 1:])

weights_table = pd.Series(w_uber, index=donor_cities).sort_values(ascending=False)
weights_table[weights_table > 0.01]

In [ ]:
pivot_uber = uber.pivot(index="city", columns="week", values="rides_thousands")
y_austin = pivot_uber.loc["Austin"].sort_index().values
y_donors = pivot_uber.loc[donor_cities].sort_index(axis=1).values
synthetic_uber = y_donors.T @ w_uber
gap_uber = y_austin - synthetic_uber

rmspe_pre_uber = np.sqrt(np.mean(gap_uber[:T0] ** 2))
tau_hat_uber = gap_uber[T0:].mean()
print(f"Pre-treatment RMSPE: {rmspe_pre_uber:.2f}")
print(f"tau_hat (mean post-treatment gap): {tau_hat_uber:.2f}")

In [ ]:
weeks = np.arange(1, len(y_austin) + 1)
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].plot(weeks, y_austin, label="Austin", color="#8e44ad", linewidth=2)
axes[0].plot(weeks, synthetic_uber, label="Control sintético", color="#e74c3c", linestyle="--", linewidth=2)
axes[0].axvline(T0 + 0.5, color="gray", linestyle=":")
axes[0].set_xlabel("Semana"); axes[0].set_ylabel("Viajes (miles)")
axes[0].set_title("Austin vs. control sintético"); axes[0].legend()

axes[1].plot(weeks, gap_uber, color="#2c3e50")
axes[1].axhline(0, color="lightgray")
axes[1].axvline(T0 + 0.5, color="gray", linestyle=":")
axes[1].set_xlabel("Semana"); axes[1].set_ylabel("Austin − Sintético")
axes[1].set_title("Brecha estimada")
plt.tight_layout(); plt.show()

**Interpretación.** El ajuste pre-tratamiento es casi exacto y los pesos se concentran en un puñado de ciudades (Kansas City, Boise, Columbus...) — la misma sparsity que se ve en el Dashboard 1 con `comparison_mode="optimized"`. $\hat\tau \approx 9.8$ está cerca del efecto verdadero simulado (10%).

### Estimador 1b — El mismo problema con `scpi_pkg`

`scpi_pkg` está diseñado alrededor de *features* (variables de resultado) y covariables de ajuste (`cov_adj`), no de una matriz de predictores estilo ADH armada a mano. Con su configuración por defecto (sin especificar `features`), usa **toda la trayectoria pre-tratamiento del resultado** como predictor — acá, las 20 semanas completas — en vez del conjunto puntual de 8 predictores (demografía + 4 semanas rezagadas) que arma el Dashboard 1.

**Nota importante:** por eso, no hay que esperar que el vector de pesos de `scpi_pkg` reproduzca exactamente el de la celda anterior — están resolviendo especificaciones distintas del problema de matching, aunque ambas son formas razonables de implementar control sintético. Este es un buen ejemplo de que "la misma idea, implementada con herramientas distintas" no siempre da un resultado idéntico. Además, el $V$ por defecto de `scpi_pkg` es `'separate'` (una variante de mínimos cuadrados por partes), distinto de la "selección simple" (varianza inversa) que usa el Dashboard 1.

In [ ]:
period_pre_uber = np.arange(1, T0 + 1)
period_post_uber = np.arange(T0 + 1, uber.week.max() + 1)

data_uber = scdata(df=uber, id_var="city", time_var="week", outcome_var="rides_thousands",
                    period_pre=period_pre_uber, period_post=period_post_uber,
                    unit_tr="Austin", unit_co=donor_cities, constant=False, verbose=False)
est_uber = scest(data_uber, w_constr={"name": "simplex"})

w_scpi_uber = est_uber.w.iloc[:, 0].sort_values(ascending=False)
w_scpi_uber[w_scpi_uber > 0.01]

## Parte 2 — Caso de estudio clásico: Proposition 99 en California

Ahora aplicamos el mismo pipeline a los datos reales de Abadie, Diamond & Hainmueller (2010): gasto anual en cigarrillos (`PacksPerCapita`, paquetes per cápita) para California y 38 estados donantes, 1970–2000. California aprobó la Proposition 99 (un fuerte impuesto al tabaco y campaña de control) en 1988, con efecto desde 1989.

In [ ]:
prop99 = pd.read_csv('https://raw.githubusercontent.com/rpasquini/econometrics-book-notebooks/main/es/ch4_inferencia_causal/data/california_prop99.csv', sep=';')

In [ ]:
PRE_YEARS = list(range(1970, 1989))
POST_YEARS = list(range(1989, 2001))
ALL_YEARS = PRE_YEARS + POST_YEARS
T0_PROP99 = len(PRE_YEARS)

pivot_prop99 = prop99.pivot(index="State", columns="Year", values="PacksPerCapita")
donor_states = [s for s in pivot_prop99.index if s != "California"]
prop99.head()

### Estimador 1a (repetido) — Pesos desde cero, sobre datos reales

Acá el único predictor es el propio resultado (paquetes per cápita) en cada uno de los 19 años pre-tratamiento — el enfoque clásico de ADH cuando no hay covariables adicionales.

In [ ]:
x1_prop99 = pivot_prop99.loc["California", PRE_YEARS].values
x0_prop99 = pivot_prop99.loc[donor_states, PRE_YEARS].values.T
w_prop99 = solve_weights(x1_prop99, x0_prop99)

weights_prop99 = pd.Series(w_prop99, index=donor_states).sort_values(ascending=False)
weights_prop99[weights_prop99 > 0.01]

In [ ]:
synthetic_prop99 = pivot_prop99.loc[donor_states, ALL_YEARS].values.T @ w_prop99
actual_prop99 = pivot_prop99.loc["California", ALL_YEARS].values
gap_prop99 = actual_prop99 - synthetic_prop99

rmspe_pre_prop99 = np.sqrt(np.mean(gap_prop99[:T0_PROP99] ** 2))
print(f"Pre-treatment RMSPE: {rmspe_pre_prop99:.2f}")
print(f"Gap by 2000: {gap_prop99[-1]:.1f} packs per capita")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(ALL_YEARS, actual_prop99, label="California", color="#8e44ad", linewidth=2)
ax.plot(ALL_YEARS, synthetic_prop99, label="Control sintético", color="#e74c3c", linestyle="--", linewidth=2)
ax.axvline(1988.5, color="gray", linestyle=":")
ax.set_xlabel("Año"); ax.set_ylabel("Paquetes de cigarrillos per cápita")
ax.set_title("Proposition 99: California vs. control sintético (ADH 2010)")
ax.legend(); plt.tight_layout(); plt.show()

**Reproducción del resultado clásico.** Los pesos se concentran en Utah, Montana, Nevada, Connecticut y New Hampshire — el mismo conjunto de donantes (con pesos muy similares) que reporta la Tabla 2 de ADH (2010). La brecha hacia el año 2000 (~-27 paquetes per cápita) también coincide con el resultado publicado: Proposition 99 redujo notablemente el consumo de cigarrillos en California relativo a su control sintético.

### Estimador 1b (repetido) — `scpi_pkg` sobre Prop 99

Con una sola *feature* (el resultado) y sin covariables adicionales, la especificación por defecto de `scpi_pkg` coincide con el enfoque clásico de ADH — por eso acá sí esperamos pesos muy parecidos a los de la celda anterior (a diferencia del caso de Uber/Austin, donde la demografía introducía una especificación distinta).

In [ ]:
period_pre_p99 = np.arange(1970, 1989)
period_post_p99 = np.arange(1989, 2001)

data_prop99 = scdata(df=prop99, id_var="State", time_var="Year", outcome_var="PacksPerCapita",
                      period_pre=period_pre_p99, period_post=period_post_p99,
                      unit_tr="California", unit_co=donor_states, constant=False, verbose=False)
est_prop99 = scest(data_prop99, w_constr={"name": "simplex"})

w_scpi_prop99 = est_prop99.w.iloc[:, 0].sort_values(ascending=False)
w_scpi_prop99[w_scpi_prop99 > 0.01]

## Estimador 2 — Inferencia por permutación

Reproducimos la lógica del Dashboard 3: reasignamos "tratamiento" a cada unidad donante por turno (usando las demás como su propio *pool* de donantes), calculamos $r_j = \text{RMSPE}_{post,j}/\text{RMSPE}_{pre,j}$ para cada una, y obtenemos un p-valor basado en el ranking de Austin/California dentro de esa distribución.

### Sobre la simulación de Uber/Austin

In [ ]:
def leave_one_out_placebo(pivot, all_units, predictor_fn, T0):
    """Run the leave-one-out placebo procedure for every unit; return r_j and gaps."""
    gaps, rmspe_pre, rmspe_post = {}, {}, {}
    for unit in all_units:
        donors = [u for u in all_units if u != unit]
        x1, x0 = predictor_fn(unit, donors)
        w = solve_weights(x1, x0)
        synthetic = pivot.loc[donors].values.T @ w
        actual = pivot.loc[unit].values
        gap = actual - synthetic
        gaps[unit] = gap
        rmspe_pre[unit] = np.sqrt(np.mean(gap[:T0] ** 2))
        rmspe_post[unit] = np.sqrt(np.mean(gap[T0:] ** 2))
    r = {u: rmspe_post[u] / max(rmspe_pre[u], 1e-9) for u in all_units}
    return gaps, rmspe_pre, r

all_cities = ["Austin"] + donor_cities

def uber_predictors(unit, donors):
    X = build_predictor_matrix(uber, [unit] + donors)
    return X[:, 0], X[:, 1:]

gaps_uber, rmspe_pre_uber_all, r_uber = leave_one_out_placebo(
    pivot_uber, all_cities, uber_predictors, T0)

r_series_uber = pd.Series(r_uber).sort_values(ascending=False)
p_value_uber = float((r_series_uber >= r_uber["Austin"]).mean())
print(f"Austin's rank: {list(r_series_uber.index).index('Austin') + 1} of {len(r_series_uber)}")
print(f"p-value: {p_value_uber:.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
colors = ["#8e44ad" if u == "Austin" else "#2980b9" for u in r_series_uber.index]
ax.scatter(r_series_uber.index, r_series_uber.values, c=colors)
ax.set_xticks(range(len(r_series_uber)))
ax.set_xticklabels(r_series_uber.index, rotation=45, ha="right")
ax.set_ylabel(r"$r_j = RMSPE_{post}/RMSPE_{pre}$")
ax.set_title(f"Distribución de $r_j$ (placebo) — Uber/Austin (p = {p_value_uber:.2f})")
plt.tight_layout(); plt.show()

Austin queda cerca del extremo superior de la distribución de placebos — consistente con el efecto simulado del 10% y con lo que se ve en el Dashboard 3 con `true_effect=10%` y un corte de ajuste razonable.

### Sobre Proposition 99 — cerrando el círculo con la distribución de placebos clásica de ADH

In [ ]:
def prop99_predictors(unit, donors):
    x1 = pivot_prop99.loc[unit, PRE_YEARS].values
    x0 = pivot_prop99.loc[donors, PRE_YEARS].values.T
    return x1, x0

all_states = ["California"] + donor_states
gaps_prop99, rmspe_pre_prop99_all, r_prop99 = leave_one_out_placebo(
    pivot_prop99[ALL_YEARS], all_states, prop99_predictors, T0_PROP99)

r_series_prop99 = pd.Series(r_prop99).sort_values(ascending=False)
p_value_prop99 = float((r_series_prop99 >= r_prop99["California"]).mean())
print(f"California's rank: {list(r_series_prop99.index).index('California') + 1} of {len(r_series_prop99)}")
print(f"p-value (all 39 states, no fit cutoff): {p_value_prop99:.3f}")

Aplicamos el mismo tipo de corte que usa el Dashboard 3 (excluir donantes cuyo RMSPE pre-tratamiento supera $k$ veces el de California):

In [ ]:
rmspe_pre_ca = rmspe_pre_prop99_all["California"]
for k in [2, 3, 5, float("inf")]:
    kept = [s for s in all_states if rmspe_pre_prop99_all[s] <= k * rmspe_pre_ca]
    r_kept = pd.Series({s: r_prop99[s] for s in kept}).sort_values(ascending=False)
    p = float((r_kept >= r_prop99["California"]).mean())
    label = "todos" if k == float("inf") else f"{k}x"
    print(f"cutoff={label:>6}: donantes retenidos={len(kept):>2}  p-value={p:.3f}")

**A diferencia de la simulación limpia del Dashboard 3, acá el corte no necesariamente afila el p-valor.** Missouri y Virginia — los dos estados con $r_j$ más alto que California — en realidad ajustan *mejor* que California en el período pre-tratamiento (su RMSPE pre-tratamiento es 0.27x y 0.49x el de California), así que ninguna regla de "excluir mal ajuste" los va a sacar del cálculo, sin importar qué tan estricta sea. Son estados que ajustaron bien pero tuvieron su propio movimiento grande e idiosincrático en el período post-tratamiento — un recordatorio de que el corte por RMSPE filtra placebos que nunca ajustaron bien, pero no garantiza excluir cualquier brecha post-tratamiento grande si viene de un donante con buen ajuste.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
for state in all_states:
    color = "#8e44ad" if state == "California" else "#bdc3c7"
    lw = 2.2 if state == "California" else 0.8
    zorder = 5 if state == "California" else 1
    ax.plot(ALL_YEARS, gaps_prop99[state], color=color, linewidth=lw, zorder=zorder)
ax.axhline(0, color="black", linewidth=0.8)
ax.axvline(1988.5, color="gray", linestyle=":")
ax.set_xlabel("Año"); ax.set_ylabel("Brecha (real − sintético)")
ax.set_title("Distribución de placebos: California vs. los 38 estados donantes")
plt.tight_layout(); plt.show()

Esta es, en esencia, la Figura 4 de ADH (2010): California (línea gruesa) se aparta de manera mucho más marcada que la mayoría de los placebos hacia el final del período post-tratamiento.

### Contraste: los intervalos de predicción de `scpi_pkg`

`scpi_pkg` implementa su propia inferencia vía `scpi()` — pero es un enfoque **materialmente distinto**, no una reproducción del p-valor de permutación de arriba. En vez de un ranking basado en placebos, `scpi()` construye **intervalos de predicción** (Cattaneo, Feng & Titiunik, 2021) usando una combinación de simulación y ajuste por sesgo/varianza de $\hat Y_{1t}^N$. Los dos enfoques responden preguntas relacionadas pero no intercambiables: uno pregunta "¿qué tan inusual es esta brecha comparada con placebos?"; el otro pregunta "¿cuál es el rango plausible del resultado no observado, dada la incertidumbre de la estimación?".

In [ ]:
pi_prop99 = scpi(data_prop99, w_constr={"name": "simplex"}, sims=200, cores=1, verbose=False)
ci = pi_prop99.CI_all_gaussian
ci.head()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(ALL_YEARS, actual_prop99, label="California", color="#8e44ad", linewidth=2)
ax.plot(ALL_YEARS, synthetic_prop99, label="Control sintético", color="#e74c3c", linestyle="--", linewidth=2)
ax.fill_between(POST_YEARS, ci["Lower"].values, ci["Upper"].values,
                 color="#e74c3c", alpha=0.15, label="Intervalo de predicción (scpi)")
ax.axvline(1988.5, color="gray", linestyle=":")
ax.set_xlabel("Año"); ax.set_ylabel("Paquetes de cigarrillos per cápita")
ax.set_title("Proposition 99 con intervalo de predicción de scpi_pkg")
ax.legend(); plt.tight_layout(); plt.show()

## Para explorar por tu cuenta

- Cambiá el corte de ajuste (`k`) en la sección de inferencia por permutación de Prop 99 y verificá cómo cambia el p-valor a medida que se excluyen más donantes con mal ajuste pre-tratamiento.
- Probá restringir los predictores del pase desde cero de Uber/Austin solo a demografía (sacando las 4 semanas rezagadas) y compará los pesos resultantes con los del Dashboard 1 bajo `predictor_set="demographics"`.
- Ajustá `w_constr` en `scest`/`scpi` (por ejemplo `{"name": "lasso"}` en vez de `"simplex"`) y observá cómo cambia la sparsity de los pesos.
- Con los datos de Alemania Occidental ya descargados en `data/scpi_germany.csv` (Abadie, Diamond & Hainmueller, 2015), replicá el mismo pipeline como ejercicio adicional.